# VectoVecto Tier B training (v11 - Kaggle-hardened)

Required: **GPU T4 x2 + Internet ON + DIV2K dataset added as Input**.

Kaggle-friendliness:
- Every run streams to `/kaggle/working/run.log` (survives crashes).
- Checkpoints save every 500 iters (max ~15 min of work at risk) and are written
  atomically, so a killed session can never corrupt them.
- Training stops itself at 10h (`--time-budget-min 600`) BEFORE Kaggle's ~12h kill.
- `--resume` continues from the checkpoint you upload as an Input dataset.
- A final cell evaluates PSNR/SSIM on DIV2K validation (if the valid split is attached).

Workflow each session:
1. Add the previous output checkpoint as a Kaggle dataset input.
2. Run all cells.
3. Save Version (or download `deep_sr/latest_checkpoint.pth` + `best_checkpoint.pth`
   from the Output panel) and re-upload for the next session.


In [ ]:
# Cell 1 - Clone repo + setup
import subprocess, sys, os, time

REPO_URL = 'https://github.com/DontHash/VectoVecto.git'
REPO_DIR = '/kaggle/working/VectoVecto'
LOG_PATH = '/kaggle/working/run.log'

if not os.path.isdir(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    # A re-run must train the LATEST code, not a stale clone from earlier.
    print('Repo exists -> pulling latest main...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main'], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=False)

os.chdir(REPO_DIR)
print('Repo ready. Files in repo root:')
for f in sorted(os.listdir('.'))[:20]:
    print(' ', f)

# Open a log file that we'll write everything into
log = open(LOG_PATH, 'w', buffering=1)  # line-buffered
def logline(msg):
    log.write(str(msg) + '\n'); log.flush()
logline(f'=== VectoVecto Tier B training run - {time.strftime("%Y-%m-%d %H:%M:%S")} ===')
logline(f'Python: {sys.version}')
logline(f'CWD: {os.getcwd()}')
logline('---')
print('Log opened at', LOG_PATH)


In [ ]:
# Cell 2 — Install deps + fix PyTorch for P100/T4 compat
# Kaggle's preinstalled PyTorch (2.5+cu124) dropped sm_60 (P100) support.
# We detect the GPU compute capability and reinstall cu118 PyTorch if needed
# (cu118 supports sm_37..sm_90 = works on P100 AND T4).
import subprocess, sys
def run(cmd, logpath=log):
    logpath.write(f'$ {cmd}\n'); logpath.flush()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    logpath.write(p.stdout); logpath.write(p.stderr); logpath.flush()
    return p.returncode, p.stdout, p.stderr

rc, out, err = run(f'{sys.executable} -m pip install -q tqdm')
logline('pip install tqdm done')

# Check GPU compute capability
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    logline(f'GPU: {name} (sm_{cap[0]}{cap[1]})')
    print(f'GPU: {name} (sm_{cap[0]}{cap[1]})')
    if cap[0] < 7:
        logline(f'GPU sm_{cap[0]}{cap[1]} < sm_70 -> reinstalling PyTorch cu118 for compat')
        print(f'sm_{cap[0]}{cap[1]} < sm_70 -> reinstalling PyTorch with cu118...')
        rc2, out2, err2 = run(f'{sys.executable} -m pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118 --force-reinstall')
        logline(f'cu118 install rc={rc2}')
        if rc2 != 0:
            logline(f'cu118 install FAILED:\n{err2}')
            print('FAILED to install cu118 PyTorch:', err2[-500:])
        else:
            logline('cu118 PyTorch installed OK')
            print('cu118 PyTorch installed. Reloading...')
else:
    logline('FATAL: GPU not enabled')
    log.close()
    raise RuntimeError('GPU not enabled. Settings -> Accelerator -> GPU.')

# Verify CUDA works after potential reinstall
rc, out, err = run(f'{sys.executable} -c "import torch; print(torch.__version__); x=torch.rand(2,2,device=\\\"cuda\\\"); print(x.sum().item()); print(\\\"CUDA OK\\\")"')
logline(f'CUDA verify: rc={rc}, out={out.strip()}, err={err.strip()[:200]}')
print('CUDA verify rc:', rc)
print(out)
if err:
    print('STDERR:', err[:500])
if rc != 0 or 'CUDA OK' not in out:
    logline('FATAL: CUDA still broken after reinstall')
    log.close()
    raise RuntimeError('CUDA not working. Check run.log for details.')
logline('GPU + CUDA verified OK')

In [ ]:
# Cell 3 — Locate DIV2K (logs every candidate path it tried)
import os
found_dir = None
for entry in os.listdir('/kaggle/input/'):
    base = os.path.join('/kaggle/input/', entry)
    logline(f'  /kaggle/input/{entry} exists={os.path.isdir(base)}')
    if os.path.isdir(base):
        for sub in ('DIV2K_train_HR', 'DIV2K_train_HR/DIV2K_train_HR', 'train_HR'):
            p = os.path.join(base, sub)
            if os.path.isdir(p):
                imgs = [f for f in os.listdir(p) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                logline(f'    {p} -> {len(imgs)} images')
                if imgs:
                    found_dir = p
                    break
        if found_dir: break

if found_dir is None:
    logline('FATAL: DIV2K HR not found')
    log.close()
    raise RuntimeError('DIV2K not found')
logline(f'DIV2K HR: {found_dir}')
logline(f'Images: {len(os.listdir(found_dir))}')
os.environ['DIV2K_HR_DIR'] = found_dir
out_dir = '/kaggle/working/deep_sr'
os.makedirs(out_dir, exist_ok=True)
os.environ['ARTIFACTS_DIR'] = out_dir
logline(f'ARTIFACTS_DIR={out_dir}')
print('DIV2K:', found_dir)

In [ ]:
# Cell 4 — Smoke test via train_deep_sr.py --smoke (STREAMS output live)
# v9: replaced subprocess.run (blocks+buffers) with Popen (streams line-by-line)
#     so we can see progress in real time, not after the process exits.
import subprocess, sys, time
t0 = time.time()
logline('--- SMOKE TEST START ---')
proc = subprocess.Popen([sys.executable, 'train_deep_sr.py', '--smoke', '--device', 'cuda', '--allow-cpu'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    log.write(line); log.flush()
proc.wait()
logline(f'--- SMOKE TEST END (rc={proc.returncode}, {time.time()-t0:.1f}s) ---')
print('Smoke rc:', proc.returncode)

In [ ]:
# Cell 5 - TRAINING: auto-resume, checkpoint every 500 iters, 10h budget (streams live)
# v11 Kaggle hardening:
#   - --save-every 500     -> at most ~15 min of work lost if the session dies
#   - --time-budget-min 600 -> exits cleanly before Kaggle's ~12h session kill
#   - retries once on crash (rc != 0) and resumes from the last checkpoint
#   - recursive checkpoint search: also finds deep_sr/latest_checkpoint.pth
#     inside an uploaded dataset
import subprocess, sys, time, os, shutil, glob

out_dir = '/kaggle/working/deep_sr'
os.makedirs(out_dir, exist_ok=True)

# --- Find the previous run's checkpoint anywhere under /kaggle/input ---
resume_src = None
for pat in ('**/latest_checkpoint.pth', '**/best_checkpoint.pth'):
    hits = glob.glob(os.path.join('/kaggle/input', pat), recursive=True)
    if hits:
        resume_src = sorted(hits)[0]
        break
if resume_src:
    shutil.copy(resume_src, os.path.join(out_dir, 'latest_checkpoint.pth'))
    logline(f'Copied checkpoint for resume: {resume_src}')
    print('Will resume from:', resume_src)
else:
    logline('No prior checkpoint found in inputs -> training from scratch')
    print('No checkpoint found -> training from scratch')

cmd = [sys.executable, 'train_deep_sr.py',
       '--device', 'cuda',
       '--resume',
       '--epochs', '100', '--batch', '8', '--patch', '32',
       '--iters-per-epoch', '250', '--save-every', '500',
       '--log-every', '50', '--workers', '2',
       '--time-budget-min', '600',
       '--w-perceptual', '0.5', '--w-adv', '0.02']
logline(f'Command: {" ".join(cmd)}')

def stream(cmd, logpath):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        logpath.write(line); logpath.flush()
        print(line, end='')
    p.wait()
    return p.returncode

t0 = time.time()
rc = 1
for attempt in (1, 2):
    logline(f'--- TRAINING START (attempt {attempt}) ---')
    rc = stream(cmd, log)
    logline(f'--- TRAINING END (attempt {attempt}, rc={rc}, {time.time()-t0:.1f}s) ---')
    if rc == 0:
        break
    logline(f'Training crashed (rc={rc}); retrying with --resume in 10s...')
    time.sleep(10)
print(f'\nTraining rc={rc}, total time {(time.time()-t0)/60:.1f} min')


In [ ]:
# Cell 6 - Evaluate checkpoints on DIV2K validation (PSNR/SSIM numbers)
# Skipped automatically if the attached DIV2K dataset has no valid split.
# v12: pass --ckpt explicitly (was defaulting to repo-relative artifacts/deep_sr
# and failing because training saves to /kaggle/working/deep_sr). Evals BOTH
# best and latest checkpoints so they can be compared.
import glob, os, subprocess, sys

valid = None
for pat in ('**/DIV2K_valid_HR', '**/valid_HR'):
    hits = [h for h in glob.glob(os.path.join('/kaggle/input', pat), recursive=True)
            if os.path.isdir(h) and any(f.lower().endswith(('.png', '.jpg', '.jpeg'))
                                        for f in os.listdir(h))]
    if hits:
        valid = sorted(hits)[0]
        break

if valid is None:
    logline('Eval skipped: no DIV2K_valid_HR found in /kaggle/input')
    print('Eval skipped: no validation split found')
else:
    eval_dir = '/kaggle/working/eval'
    os.makedirs(eval_dir, exist_ok=True)
    for tag in ('best', 'latest'):
        ckpt = f'/kaggle/working/deep_sr/{tag}_checkpoint.pth'
        if not os.path.exists(ckpt):
            print(f'Skipping {tag}: {ckpt} not found (no new best saved this session)')
            continue
        cmd = [sys.executable, 'eval_deep_sr.py', '--hr-dir', valid,
               '--ckpt', ckpt,
               '--num-images', '20', '--hr-size', '512', '--device', 'cuda',
               '--json', os.path.join(eval_dir, f'metrics_{tag}.json')]
        logline(f'Eval command: {" ".join(cmd)}')
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            print(line, end='')
        p.wait()
        logline(f'Eval {tag} rc={p.returncode}')
        print(f'Eval {tag} rc={p.returncode}; metrics -> {eval_dir}/metrics_{tag}.json')


In [ ]:
# Cell 7 - Close log + list outputs
import os
log.flush(); log.close()
print('=== run.log saved to /kaggle/working/run.log ===')
print('=== /kaggle/working/ contents ===')
for f in sorted(os.listdir('/kaggle/working/')):
    p = os.path.join('/kaggle/working/', f)
    if os.path.isfile(p):
        print(f'  {f}: {os.path.getsize(p)/1024:.1f} KB')
    else:
        print(f'  {f}/')
print('=== /kaggle/working/deep_sr/ (checkpoints) ===')
if os.path.isdir('/kaggle/working/deep_sr'):
    for f in sorted(os.listdir('/kaggle/working/deep_sr')):
        size_mb = os.path.getsize(os.path.join('/kaggle/working/deep_sr', f)) / 1e6
        print(f'  {f}: {size_mb:.1f} MB')
else:
    print('  (no checkpoints - training crashed earlier)')
print()
print('IMPORTANT: to continue next session, download latest_checkpoint.pth and')
print('best_checkpoint.pth from the Output panel (or Save Version), then re-upload')
print('them as a Kaggle dataset input and re-run this notebook. Training resumes.')
